In [ ]:
!pip install fastai>=2.7 scikit-image tqdm -q


In [ ]:
import os
import glob
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from skimage.color import rgb2lab, lab2rgb

import torch
from torch import nn, optim
from torchvision import transforms
from torchvision.utils import make_grid
from torch.utils.data import Dataset, DataLoader

from fastai.vision.learner import create_body
from torchvision.models.resnet import resnet18
from fastai.vision.models.unet import DynamicUnet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Dataset

Using a 13,000-image subset of COCO. Near-grayscale and fully grayscale images were removed first automatically (by comparing RGB channels pixel-wise) and then manually. The remaining images are split 80/20 into train and validation.

Images are converted to LAB color space: the L channel is the input to the generator, and the ab channels are what it needs to predict. L is normalized to [-1, 1] via `L/50 - 1`; ab channels are divided by 110 to fit in roughly [-1, 1].

See `data_preparation.ipynb` for the full dataset construction steps.


In [ ]:
class ColorizationDataset(Dataset):
    def __init__(self, paths, split='train'):
        self.split = split
        self.size = 256
        self.paths = paths

        if split == 'train':
            self.transforms = transforms.Compose([
                transforms.Resize((self.size, self.size), Image.BICUBIC),
                transforms.RandomHorizontalFlip(),
            ])
        else:
            self.transforms = transforms.Resize((self.size, self.size), Image.BICUBIC)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transforms(img)
        img = np.array(img)
        img_lab = rgb2lab(img).astype("float32")
        img_lab = transforms.ToTensor()(img_lab)
        L = img_lab[[0], ...] / 50. - 1.
        ab = img_lab[[1, 2], ...] / 110.
        return {'L': L, 'ab': ab}

    def __len__(self):
        return len(self.paths)


def make_dataloaders(paths, split='train', batch_size=16, n_workers=2):
    dataset = ColorizationDataset(paths, split)
    return DataLoader(dataset, batch_size=batch_size, num_workers=n_workers,
                      pin_memory=True, shuffle=(split == 'train'))


## Why ResNet-18 as the encoder?

The encoder is initialized with ImageNet-pretrained weights. This matters because the dataset is only ~13k images — not nearly enough to learn good feature detectors from scratch. The pretrained encoder already knows how to detect shapes, textures, and objects, which transfers well to colorization.

ResNet's residual connections also help with gradient flow during fine-tuning, keeping the encoder from drifting too far from its initialization.

The U-Net skip connections then pass encoder feature maps directly to the decoder, so fine spatial details (edges, textures) don't get lost through the bottleneck.

References:
- ResNet: https://ieeexplore.ieee.org/document/7780459
- DynamicUnet (fastai): https://docs.fast.ai/vision.models.unet.html


In [ ]:
SIZE = 256

# Option 1: U-Net with ResNet18 backbone
def build_res_unet(n_input=1, n_output=2, size=256, dropout=0.5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    try:
        from torchvision.models import ResNet18_Weights
        model = resnet18(weights='DEFAULT')
    except:
        try:
            model = resnet18(pretrained=True)
        except:
            model = resnet18(pretrained=False)
            print("warning: could not load pretrained weights")

    if n_input == 1:
        old_conv = model.conv1
        with torch.no_grad():
            new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        model.conv1 = new_conv

    body = create_body(model, cut=-2)
    net_G = DynamicUnet(body, n_output, (size, size)).to(device)
    return net_G


# Option 2: Classic U-Net (no pretrained weights)
class UnetBlock(nn.Module):
    def __init__(self, nf, ni, submodule=None, input_c=None, dropout=False,
                 innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost
        if input_c is None:
            input_c = nf

        downconv = nn.Conv2d(input_c, ni, kernel_size=4, stride=2, padding=1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = nn.BatchNorm2d(ni)
        uprelu = nn.ReLU(True)
        upnorm = nn.BatchNorm2d(nf)

        if outermost:
            upconv = nn.ConvTranspose2d(ni * 2, nf, kernel_size=4, stride=2, padding=1)
            down = [downconv]
            up = [uprelu, upconv, nn.Tanh()]
            model = down + [submodule] + up
        elif innermost:
            upconv = nn.ConvTranspose2d(ni, nf, kernel_size=4, stride=2, padding=1, bias=False)
            down = [downrelu, downconv]
            up = [uprelu, upconv, upnorm]
            model = down + up
        else:
            upconv = nn.ConvTranspose2d(ni * 2, nf, kernel_size=4, stride=2, padding=1, bias=False)
            down = [downrelu, downconv, downnorm]
            up = [uprelu, upconv, upnorm]
            if dropout:
                up += [nn.Dropout(0.5)]
            model = down + [submodule] + up

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        return torch.cat([x, self.model(x)], 1)


def build_classic_unet(n_input=1, n_output=2, size=256):
    unet_block = UnetBlock(512, 512, innermost=True)
    for _ in range(3):
        unet_block = UnetBlock(512, 512, submodule=unet_block, dropout=True)
    out_filters = [256, 128, 64]
    for out_f in out_filters:
        unet_block = UnetBlock(out_f, out_f * 2, submodule=unet_block)
    unet_block = UnetBlock(n_output, 64, input_c=n_input, submodule=unet_block, outermost=True)

    return unet_block


USE_RESNET = True  # set to False to use the classic U-Net


def build_generator(n_input=1, n_output=2, size=256, dropout=0.5):
    if USE_RESNET:
        return build_res_unet(n_input, n_output, size, dropout=dropout)
    else:
        return build_classic_unet(n_input, n_output, size)


# PatchGAN discriminator
class PatchDiscriminator(nn.Module):
    def __init__(self, input_c, num_filters=64, n_down=3):
        super().__init__()
        self.model = self.get_layers(input_c, num_filters, n_down)

    def get_layers(self, input_c, num_filters, n_down):
        model = [self.get_conv(input_c, num_filters, norm=False)]
        for i in range(n_down):
            model += [self.get_conv(num_filters * 2**i, num_filters * 2**(i+1),
                                    stride=1 if i == (n_down - 1) else 2)]
        model += [self.get_conv(num_filters * 2**n_down, 1, stride=1, norm=False, act=False)]
        return nn.Sequential(*model)

    def get_conv(self, in_channels, out_channels, kernel_size=4, stride=2,
                 padding=1, norm=True, act=True):
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=not norm)]
        if norm:
            layers.append(nn.BatchNorm2d(out_channels))
        if act:
            layers.append(nn.LeakyReLU(0.2, True))
        return nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


# GAN loss
class GANLoss(nn.Module):
    def __init__(self, gan_mode='vanilla', real_label=1.0, fake_label=0.0):
        super().__init__()
        self.register_buffer('real_label', torch.tensor(real_label))
        self.register_buffer('fake_label', torch.tensor(fake_label))
        self.loss = nn.BCEWithLogitsLoss() if gan_mode == 'vanilla' else nn.MSELoss()

    def get_labels(self, preds, target_is_real):
        labels = self.real_label if target_is_real else self.fake_label
        return labels.expand_as(preds)

    def __call__(self, preds, target_is_real):
        labels = self.get_labels(preds, target_is_real)
        return self.loss(preds, labels)


# Auxiliary losses
def total_variation_loss(img):
    # penalize large color differences between neighboring pixels
    diff_h = torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:])
    diff_v = torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :])
    return diff_h.mean() + diff_v.mean()


def contrast_loss(fake_img, real_img):
    # match the contrast (std of L channel) between generated and real
    fake_L = fake_img[:, 0:1, :, :]
    real_L = real_img[:, 0:1, :, :]
    fake_std = torch.std(fake_L.view(fake_L.size(0), -1), dim=1)
    real_std = torch.std(real_L.view(real_L.size(0), -1), dim=1)
    return torch.abs(fake_std - real_std).mean()


# Weight initialization
def init_weights(net, init='norm', gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and 'Conv' in classname:
            if init == 'norm':
                nn.init.normal_(m.weight.data, mean=0.0, std=gain)
            elif init == 'xavier':
                nn.init.xavier_normal_(m.weight.data, gain=gain)
            elif init == 'kaiming':
                nn.init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
        elif 'BatchNorm2d' in classname:
            nn.init.normal_(m.weight.data, 1., gain)
            nn.init.constant_(m.bias.data, 0.)
    net.apply(init_func)
    return net


def init_model(model, device):
    model = model.to(device)
    model = init_weights(model)
    return model


# Tracking
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.count, self.avg, self.sum = [0.] * 3

    def update(self, val, count=1):
        self.count += count
        self.sum += count * val
        self.avg = self.sum / self.count


def create_loss_meters():
    return {k: AverageMeter() for k in [
        'loss_D_fake', 'loss_D_real', 'loss_D',
        'loss_G_GAN', 'loss_G_L1', 'loss_G_TV', 'loss_G_contrast', 'loss_G',
    ]}


def update_losses(model, loss_meter_dict, count):
    for loss_name, meter in loss_meter_dict.items():
        meter.update(getattr(model, loss_name).item(), count=count)


def log_results(loss_meter_dict):
    for name, meter in loss_meter_dict.items():
        print(f"  {name}: {meter.avg:.5f}")


# Visualization
def lab_to_rgb(L, ab):
    L = (L + 1.) * 50.
    ab = ab * 110.
    Lab = torch.cat([L, ab], dim=1).permute(0, 2, 3, 1).cpu().numpy()
    rgb_imgs = []
    for img in Lab:
        img_rgb = lab2rgb(img)
        rgb_imgs.append(img_rgb)
    return np.stack(rgb_imgs, axis=0)


def visualize(model, data, save=False, save_path="colorization_result.png"):
    model.net_G.eval()
    with torch.no_grad():
        model.setup_input(data)
        model.forward()
    model.net_G.train()

    fake_color = model.fake_color.detach()
    real_color = model.ab
    L = model.L

    fake_imgs = lab_to_rgb(L, fake_color)
    real_imgs = lab_to_rgb(L, real_color)

    fig, axes = plt.subplots(3, min(5, len(L)), figsize=(15, 9))
    for i in range(min(5, len(L))):
        axes[0, i].imshow(L[i][0].cpu(), cmap='gray')
        axes[0, i].axis('off')
        axes[1, i].imshow(fake_imgs[i])
        axes[1, i].axis('off')
        axes[2, i].imshow(real_imgs[i])
        axes[2, i].axis('off')

    axes[0, 0].set_ylabel("Input", fontsize=12)
    axes[1, 0].set_ylabel("Output", fontsize=12)
    axes[2, 0].set_ylabel("Ground truth", fontsize=12)

    plt.tight_layout()
    if save:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


# Full GAN model
class MainModel(nn.Module):
    def __init__(self, net_G=None, lr_G=2e-4, lr_D=2e-4,
                 beta1=0.5, beta2=0.999, lambda_L1=100., lambda_TV=1.0, lambda_contrast=0.0):
        super().__init__()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.lambda_L1 = lambda_L1
        self.lambda_TV = lambda_TV
        self.lambda_contrast = lambda_contrast

        if net_G is None:
            self.net_G = init_model(build_res_unet(n_input=1, n_output=2, size=SIZE), self.device)
        else:
            self.net_G = net_G.to(self.device)

        self.net_D = init_model(PatchDiscriminator(input_c=3, n_down=3, num_filters=64), self.device)
        self.GANcriterion = GANLoss(gan_mode='vanilla').to(self.device)
        self.L1criterion = nn.L1Loss()
        self.opt_G = optim.Adam(self.net_G.parameters(), lr=lr_G, betas=(beta1, beta2))
        self.opt_D = optim.Adam(self.net_D.parameters(), lr=lr_D, betas=(beta1, beta2))

    def set_requires_grad(self, model, requires_grad=True):
        for p in model.parameters():
            p.requires_grad = requires_grad

    def setup_input(self, data):
        self.L = data['L'].to(self.device)
        self.ab = data['ab'].to(self.device)

    def forward(self):
        self.fake_color = self.net_G(self.L)

    def backward_D(self):
        fake_image = torch.cat([self.L, self.fake_color], dim=1)
        fake_preds = self.net_D(fake_image.detach())
        self.loss_D_fake = self.GANcriterion(fake_preds, False)

        real_image = torch.cat([self.L, self.ab], dim=1)
        real_preds = self.net_D(real_image)
        self.loss_D_real = self.GANcriterion(real_preds, True)

        self.loss_D = (self.loss_D_fake + self.loss_D_real) * 0.5
        self.loss_D.backward()

    def backward_G(self):
        fake_image = torch.cat([self.L, self.fake_color], dim=1)
        fake_preds = self.net_D(fake_image)

        self.loss_G_GAN = self.GANcriterion(fake_preds, True)
        self.loss_G_L1 = self.L1criterion(self.fake_color, self.ab) * self.lambda_L1

        self.loss_G_TV = total_variation_loss(fake_image) * self.lambda_TV

        if self.lambda_contrast > 0:
            real_image = torch.cat([self.L, self.ab], dim=1)
            self.loss_G_contrast = contrast_loss(fake_image, real_image) * self.lambda_contrast
        else:
            self.loss_G_contrast = torch.tensor(0.0, device=self.device)

        self.loss_G = self.loss_G_GAN + self.loss_G_L1 + self.loss_G_TV + self.loss_G_contrast
        self.loss_G.backward()

    def optimize(self):
        self.forward()
        self.net_D.train()
        self.set_requires_grad(self.net_D, True)
        self.opt_D.zero_grad()
        self.backward_D()
        self.opt_D.step()
        self.net_G.train()
        self.set_requires_grad(self.net_D, False)
        self.opt_G.zero_grad()
        self.backward_G()
        self.opt_G.step()


print("all definitions loaded")


## Load Dataset


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/datasets/coco_subset_16000"
NUM_IMAGES = 13000

image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
paths = []
for ext in image_extensions:
    paths.extend(glob.glob(os.path.join(DATASET_PATH, ext)))
    paths.extend(glob.glob(os.path.join(DATASET_PATH, '**', ext), recursive=True))

if not paths:
    raise RuntimeError(f"no images found in {DATASET_PATH}")

np.random.seed(123)
if len(paths) > NUM_IMAGES:
    paths = np.random.choice(paths, NUM_IMAGES, replace=False)

rand_idxs = np.random.permutation(len(paths))
train_paths = paths[rand_idxs[:int(len(paths) * 0.8)]]
val_paths   = paths[rand_idxs[int(len(paths) * 0.8):]]

print(f"train: {len(train_paths)}  val: {len(val_paths)}")


In [ ]:
BATCH_SIZE  = 16
NUM_WORKERS = 2  # set to 0 if you get DataLoader worker errors

train_dl = make_dataloaders(train_paths, split='train', batch_size=BATCH_SIZE, n_workers=NUM_WORKERS)
val_dl   = make_dataloaders(val_paths,   split='val',   batch_size=BATCH_SIZE, n_workers=NUM_WORKERS)

print(f"train batches: {len(train_dl)}  val batches: {len(val_dl)}")


In [ ]:
# Optional: spot-check that no grayscale images slipped through
# (should already be clean after data_preparation.ipynb)

# import random
#
# def is_color(path):
#     try:
#         arr = np.array(Image.open(path).convert("RGB"))
#         r, g, b = arr[:,:,0], arr[:,:,1], arr[:,:,2]
#         return not np.array_equal(r, g) or not np.array_equal(g, b)
#     except:
#         return False
#
# sample = random.sample(list(train_paths), 20)
# gray_count = sum(1 for p in sample if not is_color(p))
# print(f"grayscale in sample: {gray_count}/20")


## Stage 1: Pretrain Generator

The generator is pretrained with L1 loss only for 20 epochs (lr=1e-4). This gives it a stable initialization before adversarial training starts — without pretraining the discriminator tends to overpower the generator early on and training becomes unstable.


In [ ]:
USE_RESNET    = True
USE_DROPOUT   = True
DROPOUT_PROB  = 0.5

SAVE_TO_DRIVE = True
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/colorization_checkpoints"

if SAVE_TO_DRIVE:
    if not os.path.exists("/content/drive"):
        drive.mount('/content/drive')
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    print(f"checkpoints -> {DRIVE_CHECKPOINT_DIR}")


## Experiment Configuration

Each experiment uses a different combination of loss weights. To run experiments in parallel, open the notebook in separate Colab tabs and change `EXPERIMENT_NAME` in each.

| Name | lambda_TV | lambda_contrast |
|---|---|---|
| baseline | 0.0 | 0.0 |
| tv_only | 1.0 | 0.0 |
| tv_contrast | 1.0 | 5.0 |
| contrast_only | 0.0 | 5.0 |

Checkpoints go into separate Drive folders so the runs don't interfere with each other. All experiments can reuse the same pretrained generator from Stage 1 — checkpoints only store weights, not loss configuration.


In [ ]:
EXPERIMENT_NAME = "baseline"  # change per tab

configs = {
    "baseline":      {"lambda_L1": 100.0, "lambda_TV": 0.0, "lambda_contrast": 0.0},
    "tv_only":       {"lambda_L1": 100.0, "lambda_TV": 1.0, "lambda_contrast": 0.0},
    "tv_contrast":   {"lambda_L1": 100.0, "lambda_TV": 1.0, "lambda_contrast": 5.0},
    "contrast_only": {"lambda_L1": 100.0, "lambda_TV": 0.0, "lambda_contrast": 5.0},
}

cfg = configs.get(EXPERIMENT_NAME, configs["baseline"])
LAMBDA_L1       = cfg["lambda_L1"]
LAMBDA_TV       = cfg["lambda_TV"]
LAMBDA_CONTRAST = cfg["lambda_contrast"]

if EXPERIMENT_NAME != "baseline":
    DRIVE_CHECKPOINT_DIR = f"/content/drive/MyDrive/colorization_checkpoints_{EXPERIMENT_NAME}"
    if SAVE_TO_DRIVE and os.path.exists("/content/drive"):
        os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

print(f"experiment:       {EXPERIMENT_NAME}")
print(f"lambda_L1:        {LAMBDA_L1}")
print(f"lambda_TV:        {LAMBDA_TV}")
print(f"lambda_contrast:  {LAMBDA_CONTRAST}")
print(f"checkpoint dir:   {DRIVE_CHECKPOINT_DIR}")


In [ ]:
def pretrain_generator(net_G, train_dl, opt, criterion, epochs, device, start_epoch=0):
    for e in range(start_epoch, epochs):
        loss_meter = AverageMeter()
        net_G.train()

        for data in tqdm(train_dl, desc=f"epoch {e+1}/{epochs}"):
            L  = data['L'].to(device)
            ab = data['ab'].to(device)
            preds = net_G(L)
            loss  = criterion(preds, ab)
            opt.zero_grad()
            loss.backward()
            opt.step()
            loss_meter.update(loss.item(), L.size(0))

        print(f"epoch {e+1}/{epochs}  l1: {loss_meter.avg:.5f}")

        if (e + 1) % 5 == 0 or (e + 1) == epochs:
            ckpt = os.path.join(DRIVE_CHECKPOINT_DIR, f"pretrain_epoch_{e+1}.pth") if SAVE_TO_DRIVE                    else f"pretrain_epoch_{e+1}.pth"
            torch.save({'epoch': e+1, 'net_G': net_G.state_dict(), 'opt': opt.state_dict()}, ckpt)
            print(f"  saved {ckpt}")

    return net_G


net_G = build_generator(n_input=1, n_output=2, size=SIZE,
                        dropout=DROPOUT_PROB if USE_DROPOUT else 0.0)

# load existing pretrained generator if available (saves ~1 hour of Stage 1)
pretrained_paths = [
    "/content/drive/MyDrive/colorization_checkpoints/pretrained_generator.pth",
    os.path.join(DRIVE_CHECKPOINT_DIR, "pretrained_generator.pth"),
    "pretrained_generator.pth",
]
SKIP_PRETRAINING = False
for p in pretrained_paths:
    if os.path.exists(p):
        net_G.load_state_dict(torch.load(p, map_location=device))
        print(f"loaded pretrained generator from {p}")
        SKIP_PRETRAINING = True
        break

if not SKIP_PRETRAINING:
    opt_pretrain = optim.Adam(net_G.parameters(), lr=1e-4)
    net_G = pretrain_generator(net_G, train_dl, opt_pretrain, nn.L1Loss(), epochs=20, device=device)
    save_path = os.path.join(DRIVE_CHECKPOINT_DIR, "pretrained_generator.pth") if SAVE_TO_DRIVE                 else "pretrained_generator.pth"
    torch.save(net_G.state_dict(), save_path)
    print(f"pretrained generator saved to {save_path}")


## Stage 2: Adversarial Training

The full GAN trains for 20 epochs (lr=2e-4, Adam with beta1=0.5). The discriminator is updated first each iteration, then the generator. The generator loss combines L1 + adversarial + TV + (optionally) contrast.


In [ ]:
# Resume from checkpoint (optional - only if training was interrupted)

RESUME_TRAINING      = False
CHECKPOINT_TO_LOAD   = "checkpoint_epoch_10.pth"
START_EPOCH          = 10

if RESUME_TRAINING:
    ckpt_path = None
    for d in [DRIVE_CHECKPOINT_DIR, "/content/drive/MyDrive/colorization_checkpoints", "."]:
        candidate = os.path.join(d, CHECKPOINT_TO_LOAD)
        if os.path.exists(candidate):
            ckpt_path = candidate
            break

    if ckpt_path:
        ckpt = torch.load(ckpt_path, map_location=device)
        net_G = build_generator(n_input=1, n_output=2, size=SIZE,
                                dropout=DROPOUT_PROB if USE_DROPOUT else 0.0)
        net_G.load_state_dict(ckpt['generator_state_dict'])
        model = MainModel(net_G=net_G, lambda_L1=LAMBDA_L1, lambda_TV=LAMBDA_TV,
                         lambda_contrast=LAMBDA_CONTRAST)
        if 'optimizer_G_state_dict' in ckpt:
            model.opt_G.load_state_dict(ckpt['optimizer_G_state_dict'])
            model.opt_D.load_state_dict(ckpt['optimizer_D_state_dict'])
        print(f"resumed from epoch {ckpt['epoch']}")
    else:
        print(f"checkpoint not found: {CHECKPOINT_TO_LOAD}")
        RESUME_TRAINING = False


In [ ]:
def train_model(model, train_dl, val_dl, epochs, display_every=200, start_epoch=0):
    val_data = next(iter(val_dl))

    for e in range(start_epoch, epochs):
        loss_meter_dict = create_loss_meters()
        i = 0

        for data in tqdm(train_dl, desc=f"epoch {e+1}/{epochs}"):
            model.setup_input(data)
            model.optimize()
            update_losses(model, loss_meter_dict, count=data['L'].size(0))
            i += 1

            if i % display_every == 0:
                print(f"\nepoch {e+1}  iter {i}/{len(train_dl)}")
                log_results(loss_meter_dict)
                visualize(model, val_data, save=True)

        if (e + 1) % 5 == 0 or (e + 1) == epochs:
            ckpt = os.path.join(DRIVE_CHECKPOINT_DIR, f"checkpoint_epoch_{e+1}.pth") if SAVE_TO_DRIVE                    else f"checkpoint_epoch_{e+1}.pth"
            torch.save({
                'epoch': e + 1,
                'generator_state_dict': model.net_G.state_dict(),
                'discriminator_state_dict': model.net_D.state_dict(),
                'optimizer_G_state_dict': model.opt_G.state_dict(),
                'optimizer_D_state_dict': model.opt_D.state_dict(),
            }, ckpt)
            print(f"  saved {ckpt}")


if not RESUME_TRAINING:
    model = MainModel(net_G=net_G, lambda_L1=LAMBDA_L1, lambda_TV=LAMBDA_TV,
                     lambda_contrast=LAMBDA_CONTRAST)

GAN_EPOCHS = 20
train_model(model, train_dl, val_dl, epochs=GAN_EPOCHS,
            start_epoch=START_EPOCH if RESUME_TRAINING else 0)

# save final model
final_path = os.path.join(DRIVE_CHECKPOINT_DIR, "final_model.pth") if SAVE_TO_DRIVE else "final_model.pth"
torch.save({
    'generator_state_dict': model.net_G.state_dict(),
    'discriminator_state_dict': model.net_D.state_dict(),
}, final_path)
print(f"done. model saved to {final_path}")


## Inference


In [ ]:
# model.net_G.load_state_dict(torch.load('final_model.pth', map_location=device)['generator_state_dict'])
model.net_G.eval()

test_image_path = "/content/drive/MyDrive/datasets/test_image.jpg"

if os.path.exists(test_image_path):
    img = Image.open(test_image_path).convert("RGB")
    img = img.resize((SIZE, SIZE), Image.BICUBIC)
    img_lab = rgb2lab(np.array(img)).astype("float32")
    img_lab = transforms.ToTensor()(img_lab)
    L = (img_lab[[0], ...] / 50. - 1.).unsqueeze(0).to(device)

    with torch.no_grad():
        ab = model.net_G(L)

    colorized = lab_to_rgb(L.cpu(), ab.cpu())[0]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.convert('L'), cmap='gray')
    axes[0].set_title("input")
    axes[0].axis('off')
    axes[1].imshow(colorized)
    axes[1].set_title("colorized")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"image not found: {test_image_path}")
